# Demo: flujo de delegación del orquestador multi-agente

Ejecutá las celdas en orden. Necesitás un `.env` con tu API key (ver `README.md`).

La consulta está elegida para que ningún especialista pueda resolverla solo: hace falta
**buscar** el SLA y las métricas, y después **calcular** percentiles y la brecha.


## 1. Entorno y grafo


In [ ]:
from dotenv import load_dotenv
load_dotenv()

from graph import compile_graph, MAX_PASOS

grafo = compile_graph()
print('tope de pasos del supervisor:', MAX_PASOS)


## 2. Topología del grafo

Las aristas punteadas salen del supervisor: son las condicionales.


In [ ]:
from IPython.display import Image, display

# Requiere red (usa el renderer de mermaid.ink). Si falla, mirá grafo.mmd.
try:
    display(Image(grafo.get_graph().draw_mermaid_png()))
except Exception as e:
    print('no se pudo renderar el PNG:', e)
    print(grafo.get_graph().draw_mermaid())


## 3. El flujo de delegación, paso a paso

`astream` emite un evento por nodo, así que se ve a quién delega el supervisor en
cada vuelta y con qué motivo — no sólo el resultado final.


In [ ]:
from langchain_core.messages import HumanMessage

CONSULTA = ('¿Cuál de nuestros servicios tiene peor latencia y cuánto tendría '
            'que mejorar para cumplir el SLA vigente?')

estado = {'messages': [HumanMessage(content=CONSULTA)], 'contribuciones': [], 'pasos': 0}

async for evento in grafo.astream(estado, {'recursion_limit': 25}):
    for nodo, salida in evento.items():
        print()
        if nodo == 'supervisor':
            print(f"[supervisor] -> {salida['next_agent']}")
            print(f"             motivo: {salida['veredicto']}")
        elif nodo == 'sintesis':
            print('-' * 70)
            print('RESPUESTA FINAL:', salida['messages'][-1].content)
        else:
            print(f"[{nodo}] aporta:")
            print('  ' + salida['contribuciones'][0]['resumen'][:400])


## 4. El estado compartido

`contribuciones` es lo que permite saber quién aportó qué sin releer todo el historial.


In [ ]:
final = await grafo.ainvoke(estado, {'recursion_limit': 25})

print('vueltas del supervisor:', final['pasos'])
print('veredicto final:', final['veredicto'])
for c in final['contribuciones']:
    print(f"- [{c['agente']}] {c['resumen'][:120]}...")


## 5. El freno contra el bucle infinito

Además de la rúbrica en el prompt, hay un tope duro: si el supervisor llega a
`MAX_PASOS` vueltas, se fuerza el cierre y se sintetiza con lo que haya. Eso no
depende de que el modelo se porte bien.


In [ ]:
print(final['veredicto'])
print(f"pasos usados: {final['pasos']} / {MAX_PASOS}")
